# Projet 1 : Détection de Fraude Bancaire 🏦
## Phase 1 : Exploration et Nettoyage des Données (EDA)

**Objectif :** Comprendre la structure du dataset de transactions réelles, analyser la distribution des fraudes et préparer les données pour nos futures analyses SQL et Power BI.

---
### 1. Importation des bibliothèques nécessaires

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Bibliothèques chargées avec succès !")

Bibliothèques chargées avec succès !


In [2]:
# Chargement du fichier CSV réel
df = pd.read_csv('../data/creditcard.csv')

# Affichage des 5 premières lignes pour vérifier que tout est bien chargé
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## Check technique du Dataset

In [3]:
# 1. Afficher le nombre de lignes et de colonnes
print(f"Dimensions du dataset : {df.shape[0]} lignes et {df.shape[1]} colonnes\n")

# 2. Obtenir un résumé technique (types de données et valeurs manquantes)
df.info()

Dimensions du dataset : 284807 lignes et 31 colonnes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64

## Analyse Métier – Le déséquilibre des classes

In [5]:
# Compter le nombre de transactions normales vs frauduleuses
print("Répartition des transactions (0 = Normal, 1 = Fraude) :")
print(df['Class'].value_counts())

# Afficher les proportions en pourcentage
print("\nProportion en % :")
print(df['Class'].value_counts(normalize=True) * 100)

Répartition des transactions (0 = Normal, 1 = Fraude) :
Class
0    284315
1       492
Name: count, dtype: int64

Proportion en % :
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


## Analyse statistique des montants

In [6]:
# Statistiques pour les transactions normales
print("--- TRANSACTIONS NORMALES ---")
print(df[df['Class'] == 0]['Amount'].describe())

# Statistiques pour les transactions frauduleuses
print("\n--- TRANSACTIONS FRAUDULEUSES ---")
print(df[df['Class'] == 1]['Amount'].describe())

--- TRANSACTIONS NORMALES ---
count    284315.000000
mean         88.291022
std         250.105092
min           0.000000
25%           5.650000
50%          22.000000
75%          77.050000
max       25691.160000
Name: Amount, dtype: float64

--- TRANSACTIONS FRAUDULEUSES ---
count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64


## Préparation des données pour SQL et Power BI

In [11]:
# Sélectionner uniquement les colonnes utiles pour la suite du projet
df_clean = df[['Time', 'Amount', 'Class']]

# Sauvegarder ce dataset épuré dans notre dossier 'data' pour SQL et Power BI
df_clean.to_csv('../data/transactions_clean.csv', index=False)

print("Fichier 'transactions_clean.csv' créé avec succès dans le dossier data/ !")

Fichier 'transactions_clean.csv' créé avec succès dans le dossier data/ !


---
## 2. Simulation de la Base de Données (SQL)
**Objectif :** Charger nos données nettoyées dans une table SQL et exécuter des requêtes d'analyse métier pour extraire les premiers indicateurs de fraude.

## Créer la table SQL et charger les données

In [8]:
import sqlite3

# 1. Connexion à la base de données (elle se crée automatiquement si elle n'existe pas)
conn = sqlite3.connect('../scripts_sql/banque_fraude.db')
cursor = conn.cursor()

# 2. Charger notre DataFrame propre directement dans une table SQL nommée 'transactions'
df_clean.to_sql('transactions', conn, if_exists='replace', index=False)

# 3. Vérification : compter le nombre de lignes dans la table SQL
cursor.execute("SELECT COUNT(*) FROM transactions;")
nb_lignes = cursor.fetchone()[0]

print(f"Base de données initialisée ! La table 'transactions' contient {nb_lignes} lignes.")

# Fermeture temporaire de la connexion
conn.close()

Base de données initialisée ! La table 'transactions' contient 284807 lignes.


In [9]:
# Reconnexion à notre base SQL
conn = sqlite3.connect('../scripts_sql/banque_fraude.db')

# Requête SQL pour calculer les indicateurs clés de la fraude
query = """
SELECT 
    COUNT(*) as total_transactions,
    SUM(Class) as total_fraudes,
    ROUND(AVG(Class) * 100, 4) as taux_fraude_pourcentage,
    ROUND(SUM(CASE WHEN Class = 1 THEN Amount ELSE 0 END), 2) as montant_total_fraude
FROM 
    transactions;
"""

# Exécution de la requête et affichage sous forme de tableau propre
df_res = pd.read_sql_query(query, conn)
conn.close()

df_res

,total_transactions,total_fraudes,taux_fraude_pourcentage,montant_total_fraude
0,284807,492,0.1727,60127.97


## Analyse SQL avancée – Segmentation par montants

In [10]:
# Reconnexion à la base SQL
conn = sqlite3.connect('../scripts_sql/banque_fraude.db')

# Requête SQL avec segmentation (CASE WHEN) et regroupement (GROUP BY)
query_segments = """
SELECT 
    CASE 
        WHEN Amount <= 10 THEN '01. Petits montants (<= 10€)'
        WHEN Amount > 10 AND Amount <= 100 THEN '02. Montants moyens (10€ - 100€)'
        WHEN Amount > 100 AND Amount <= 1000 THEN '03. Grands montants (100€ - 1000€)'
        ELSE '04. Montants critiques (> 1000€)'
    END as tranche_montant,
    COUNT(*) as total_transactions,
    SUM(Class) as total_fraudes,
    ROUND(SUM(CASE WHEN Class = 1 THEN Amount ELSE 0 END), 2) as montant_total_fraude
FROM 
    transactions
GROUP BY 
    tranche_montant
ORDER BY 
    tranche_montant;
"""

df_segments = pd.read_sql_query(query_segments, conn)
conn.close()

df_segments

,tranche_montant,total_transactions,total_fraudes,montant_total_fraude
0,01. Petits montants (<= 10€),100264,249,458.90
1,02. Montants moyens (10€ - 100€),128035,113,6560.36
2,03. Grands montants (100€ - 1000€),53568,121,39871.38
3,04. Montants critiques (> 1000€),2940,9,13237.33
